<a href="https://colab.research.google.com/github/amoeba-aoi/ImmuScope_reproduction_and_extension/blob/main/ImmuScope_ablation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1 挂载 Drive，用于备份
from pathlib import Path
import os

print('1 use Drive')
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

# 2 若不存在仓库则从github克隆
print('2 clone github')
%cd /content
if not Path("/content/ImmuScope").exists():
    !git clone https://github.com/shenlongchen/ImmuScope.git
%cd /content/ImmuScope

# 3 安装依赖
print('3 install dependency')
!pip install -q --force-reinstall --no-cache-dir \
  "numpy==1.26.4" \
  "pandas==2.2.2" \
  "scikit-learn==1.4.2" \
  "h5py==3.11.0" \
  "click==8.0.4"\
  tqdm \
  ruamel.yaml

import numpy, pandas, sklearn, h5py, click
print("numpy", numpy.__version__)
print("pandas", pandas.__version__)
print("sklearn", sklearn.__version__)
print("h5py", h5py.__version__)
print("click", click.__version__)

# 4 优先从 Drive 恢复数据与权重，不存在则下载
print('4 recover data and weight')
import shutil

DRIVE_BACKUP_DIR = Path("/content/drive/MyDrive/ImmuScope_backup")  # 可改成你的实际备份目录
DRIVE_BACKUP_DIR.mkdir(parents=True, exist_ok=True)

artifacts = {
    "ImmuScope-data.tar.gz": "https://zenodo.org/records/14810445/files/ImmuScope-data.tar.gz?download=1",
    "ImmuScope-weights.tar.gz": "https://zenodo.org/records/14810445/files/ImmuScope-weights.tar.gz?download=1",
}

def gzip_ok(fp: Path) -> bool:
    if (not fp.exists()) or fp.stat().st_size == 0:
        return False
    # 返回码 0 表示 gzip 完整
    return os.system(f'gzip -t "{fp}" >/dev/null 2>&1') == 0

for name, url in artifacts.items():
    local_fp = Path(name)
    drive_fp = DRIVE_BACKUP_DIR / name

    # A) 本地包完整 -> 直接用
    if gzip_ok(local_fp):
        print(f"[OK] local exists: {local_fp}")
        continue

    # B) 本地包损坏则删除
    if local_fp.exists():
        local_fp.unlink()
        print(f"[CLEAN] removed broken local: {local_fp}")

    # C) Drive 包完整 -> 恢复到本地
    if gzip_ok(drive_fp):
        shutil.copy2(drive_fp, local_fp)
        print(f"[RESTORE] from Drive: {drive_fp}")
    else:

        # D) Drive 不存在可用包 -> 下载
        print(f"[DOWNLOAD] {name}")
        !wget -c "{url}" -O "{name}"

        if not gzip_ok(local_fp):
            raise RuntimeError(f"{name} 下载后仍不完整，请重试。")

# 再次完整性测试
!gzip -t ImmuScope-data.tar.gz && echo "data tar.gz OK"
!gzip -t ImmuScope-weights.tar.gz && echo "weights tar.gz OK"

# 查看包内结构
!tar -tzf ImmuScope-data.tar.gz | head -n 20
!tar -tzf ImmuScope-weights.tar.gz | head -n 20

# 解压
!tar -xzf ImmuScope-data.tar.gz -C .
!tar -xzf ImmuScope-weights.tar.gz -C .

# 快速确认
!ls -lah data/raw | head
!ls -lah data/train_test_h5py | head

1 use Drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
2 clone github
/content
/content/ImmuScope
3 install dependency
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 144.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 284.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 262.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 364.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 346.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 284.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 245.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.5/97.5 kB 278.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 290.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 402.9 MB/s eta 0

In [ ]:
# 5 写 configs/data.yaml
config_content = """mhc_seq: data/raw/pseudosequence.2023.dat
dataset_id_mhc: data/raw/allelelist
dataset_ms: data/tmp_datasets
5cv_ma: data/5cv_MA_h5py/5cv_.h5
5cv_sa: data/5cv_SA_h5py/5cv_.h5
train_sa: data/train_test_h5py/sa_train.h5
train_ma: data/train_test_h5py/ma_train.h5
train_ba: data/train_test_h5py/ba_train.h5
test: data/train_test_h5py/NetMHCIIpan_eval.h5
train_imm: data/imm/imm_train.h5
test_imm: data/imm/imm_test.h5
logs: results/logs
results: results
"""
Path("configs").mkdir(parents=True, exist_ok=True)
with open("configs/data.yaml", "w") as f:
    f.write(config_content)

print("configs/data.yaml 已写入")
!ls -lah data/train_test_h5py/

# main_antigen_presentation_train.py 中
# train_path_ms = Path(os.path.join(data_cnf["dataset_ms"], f"{model_name}_{model_id}_train.h5"))
# res_path_with_id = Path(res_path, f'{model_name}-{model_id}')
# 其他目录在代码中有显式创建，但dataset_ms没有使用mkdir创建目录，导致往其中写入文件时报错
# 修复代码，可作为工程改进点
# reproducibility / robustness enhancement（提升可复现性和稳定性）
!mkdir -p data/tmp_datasets
!ls -ld data/tmp_datasets

# 快速验证 configs/data.yaml 路径是否可用
from pathlib import Path
import yaml

cfg_path = Path("configs/data.yaml")
assert cfg_path.exists(), f"未找到配置文件: {cfg_path}"

with open(cfg_path, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

print(f"读取配置: {cfg_path}\n")

# 1) 先确保日志目录存在
Path(cfg.get("logs", "results/logs")).mkdir(parents=True, exist_ok=True)
Path(cfg.get("dataset_ms", "data/tmp_datasets")).mkdir(parents=True, exist_ok=True)

# 2) 常规路径检查
skip_keys = {"5cv_ma", "5cv_sa"}
ok, bad = [], []
for k, v in cfg.items():
    if k in skip_keys:
        continue
    p = Path(v)
    if p.exists():
        ok.append((k, v))
    else:
        bad.append((k, v))

print("=== 常规路径检查 ===")
for k, v in ok:
    print(f"[OK ] {k:12s} -> {v}")
for k, v in bad:
    print(f"[MISS] {k:12s} -> {v}")

# 3) 5cv 专项检查：检查展开后的真实文件
print("\n=== 5CV 文件检查 ===")
cv_bad = []
for key in ["5cv_ma", "5cv_sa"]:
    templ = cfg.get(key)
    if not templ:
        cv_bad.append((key, "未配置"))
        continue

    # 期望模板形如 .../5cv_.h5
    files_to_check = []
    if key == "5cv_ma":
        files_to_check = [templ.replace("_.h5", f"_{i}_train.h5") for i in range(5)]
    else:  # 5cv_sa
        files_to_check = (
            [templ.replace("_.h5", f"_{i}_train.h5") for i in range(5)] +
            [templ.replace("_.h5", f"_{i}_test.h5") for i in range(5)]
        )

    missing = [p for p in files_to_check if not Path(p).exists()]
    if missing:
        cv_bad.append((key, missing[:3]))  # 只展示前3个
        print(f"[MISS] {key} 缺失 {len(missing)} 个文件，例如: {missing[:3]}")
    else:
        print(f"[OK ] {key} 5-fold 文件齐全")

# 4) 关键文件检查
critical = [
    "data/raw/pseudosequence.2023.dat",
    "data/raw/allelelist",
    "data/train_test_h5py/sa_train.h5",
    "data/train_test_h5py/ma_train.h5",
    "data/train_test_h5py/ba_train.h5",
    "data/train_test_h5py/NetMHCIIpan_eval.h5",
    "data/imm/imm_train.h5",
    "data/imm/imm_test.h5",
]
print("\n=== 关键文件检查 ===")
critical_bad = []
for p in critical:
    exists = Path(p).exists()
    print(f"{'OK  ' if exists else 'MISS'} {p}")
    if not exists:
        critical_bad.append(p)

# 5) 结果输出
if bad or critical_bad:
    raise FileNotFoundError(
        f"常规缺失 {len(bad)} 项, 关键缺失 {len(critical_bad)} 项，请修正后再训练。"
    )
else:
    print("\n常规训练路径验证通过。")

if cv_bad:
    print("\n提示：5cv 相关文件不完整，当前不适合跑 main_antigen_presentation_5cv.py")
else:
    print("5cv 路径验证通过，可跑 5cv。")

configs/data.yaml 已写入
total 4.0G
drwxrwxr-x 3 1000 1000 4.0K Nov 18  2024 .
drwxrwxr-x 9 1000 1000 4.0K Nov 18  2024 ..
-rw-rw-r-- 1 1000 1000  26M Nov 17  2024 ba_train.h5
-rw-rw-r-- 1 1000 1000 3.3G Nov 17  2024 ma_train.h5
drwxrwxr-x 2 1000 1000 4.0K Nov 19  2024 ms_tmp_dataset
-rw-rw-r-- 1 1000 1000 164M Nov 17  2024 NetMHCIIpan_eval.h5
-rw-rw-r-- 1 1000 1000 545M Nov 17  2024 sa_train.h5
drwxr-xr-x 2 root root 4096 Apr  6 05:21 data/tmp_datasets
读取配置: configs/data.yaml

=== 常规路径检查 ===
[OK ] mhc_seq      -> data/raw/pseudosequence.2023.dat
[OK ] dataset_id_mhc -> data/raw/allelelist
[OK ] dataset_ms   -> data/tmp_datasets
[OK ] train_sa     -> data/train_test_h5py/sa_train.h5
[OK ] train_ma     -> data/train_test_h5py/ma_train.h5
[OK ] train_ba     -> data/train_test_h5py/ba_train.h5
[OK ] test         -> data/train_test_h5py/NetMHCIIpan_eval.h5
[OK ] train_imm    -> data/imm/imm_train.h5
[OK ] test_imm     -> data/imm/imm_test.h5
[OK ] logs         -> results/logs
[OK ] results   

In [ ]:
import psutil

ram_gb = psutil.virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))

if ram_gb < 20:
  print('Not using a high-RAM runtime')
else:
  print('You are using a high-RAM runtime!')

Your runtime has 54.8 gigabytes of available RAM

You are using a high-RAM runtime!


In [ ]:
# 修改main_antigen_presentation_train.py 中train集过大问题
# create_splits_train_valid_test函数实现：train 用 train_ratio，valid 用 valid_ratio，test 用剩下全部
# 修复代码，可作为工程改进点
# reproducibility / robustness enhancement
from pathlib import Path

p = Path("/content/ImmuScope/main_antigen_presentation_train.py")
s = p.read_text()
s = s.replace("train_ratio=0.1, valid_ratio=0.05,",
              "train_ratio=0.9, valid_ratio=0.05,")
p.write_text(s)
print("patched:", p)

patched: /content/ImmuScope/main_antigen_presentation_train.py


In [ ]:
# 修改main_antigen_presentation_train.py 中最小运行模式中修改了参数fine_tune_epochs = 0 导致要找fine-tune-b.pt的情况
# 强制在脚本末尾的测试 / output_res 总是加载 -pretrain.pt，而不会加载 -fine-tune-b.pt
# fine_tune_b >0 时不适用
%cd /content/ImmuScope
from pathlib import Path
import re

p = Path("main_antigen_presentation_train.py")
s = p.read_text()

# 强制重写两个测试函数
s = re.sub(
    r"def test_immuscope_el\(trainer, model_cnf, test_path, mhc_name_seq\):[\s\S]*?def test_immuscope_el_with_loader",
    """def test_immuscope_el(trainer, model_cnf, test_path, mhc_name_seq):
    test_loader = DataLoader(SinInstanceBag(test_path, mhc_name_seq, indices=None),
                             batch_size=model_cnf['test']['batch_size'])
    pred_instances, pred_bags, _ = trainer.predict(test_loader, model_prefix='pretrain')
    return pred_instances, pred_bags, test_loader.dataset.labels, test_loader.dataset.mhc_names

def test_immuscope_el_with_loader""",
    s
)

s = re.sub(
    r"def test_immuscope_el_with_loader\(trainer, test_loader\):[\s\S]*?@click\.command\(\)",
    """def test_immuscope_el_with_loader(trainer, test_loader):
    pred_instances, pred_bags, _ = trainer.predict(test_loader, model_prefix='pretrain')
    return (pred_instances, pred_bags, test_loader.dataset.labels[test_loader.dataset.indices],
            test_loader.dataset.mhc_names[test_loader.dataset.indices])


@click.command()""",
    s
)

p.write_text(s)
print("patched")

# 语法检查
!python -m py_compile /content/ImmuScope/main_antigen_presentation_train.py && echo "syntax ok"

/content/ImmuScope
patched
syntax ok


In [ ]:
# 恢复原main_antigen_presentation_train.py
# 去除划分比例修改和正则补丁，划分比例main和data_utils的不一致在后续论文中说明
%cd /content/ImmuScope
!git checkout -- main_antigen_presentation_train.py

# 重新做data/tmp_datasets的显式创建
!mkdir -p data/tmp_datasets
!ls -ld data/tmp_datasets

In [ ]:
# EL 单任务消融
# yaml修改
%cd /content/ImmuScope
import yaml

file_name = ("ImmuScope-EL-no-si", "ImmuScope-EL-no-si-no-ftb", "ImmuScope-EL-no-si-no-ftb-A3-supcon", "ImmuScope-EL-no-si-no-ftb-A4-no-metric")

with open("configs/ImmuScope-EL.yaml") as f:
    c = yaml.safe_load(f)

c["name"] = file_name[0]
c["train"]["sample_incorporate_epochs"] = 0

with open("configs/ImmuScope-EL-no-si.yaml", "w") as f:
    yaml.safe_dump(c, f, sort_keys=False)
print("Wrote", file_name[0])

with open("configs/ImmuScope-EL-no-si.yaml") as f:
    c = yaml.safe_load(f)

c["name"] = file_name[1]
c["train"]["fine_tune_epochs"] = 0

with open("configs/ImmuScope-EL-no-si-no-ftb.yaml", "w") as f:
    yaml.safe_dump(c, f, sort_keys=False)
print("Wrote", file_name[1])

with open("configs/ImmuScope-EL-no-si-no-ftb.yaml") as f:
    c = yaml.safe_load(f)

c["name"] = file_name[2]
c["train"]["metric_fn"] = "Contrastive"

with open("configs/ImmuScope-EL-no-si-no-ftb-A3-supcon.yaml", "w") as f:
    yaml.safe_dump(c, f, sort_keys=False)
print("Wrote", file_name[2])

with open("configs/ImmuScope-EL-no-si-no-ftb-A3-supcon.yaml") as f:
    c = yaml.safe_load(f)
c["name"] = file_name[3]
c["train"]["metric_fn"] = None

with open("configs/ImmuScope-EL-no-si-no-ftb-A4-no-metric.yaml", "w") as f:
    yaml.safe_dump(c, f, sort_keys=False)
print("Wrote", file_name[3])

/content/ImmuScope
Wrote ImmuScope-EL-no-si
Wrote ImmuScope-EL-no-si-no-ftb
Wrote ImmuScope-EL-no-si-no-ftb-A3-supcon
Wrote ImmuScope-EL-no-si-no-ftb-A4-no-metric


In [ ]:
# 前版本错误将去除SI和FTB的版本命名为no-si，前缀替换：ImmuScope-EL-no-si -> ImmuScope-EL-no-si-no-ftb
%cd /content/ImmuScope

import os
from pathlib import Path

OLD = "ImmuScope-EL-no-si"
NEW = "ImmuScope-EL-no-si-no-ftb"
ROOT = Path(".").resolve()

print("CWD:", os.getcwd())
print("ROOT:", ROOT)

# 看实际有哪些「像」相关的文件（方便核对前缀是否写对）
candidates = sorted(ROOT.rglob("*"))
related = [p for p in candidates if p.is_file() and "A2-no-si" in p.name]
print(f"Files with 'A2-no-si' in name: {len(related)}")
for p in related[:50]:
    print(" ", p.relative_to(ROOT))
if len(related) > 50:
    print("  ...")

matches = sorted(p for p in ROOT.rglob("*") if p.is_file() and p.name.startswith(OLD))
print(f"\nFiles with name.startswith({OLD!r}): {len(matches)}")
for p in matches:
    print(" ", p.relative_to(ROOT))

# 必须为 False 才会真正 rename
DO_RENAME = True

if not matches:
    print("\n没有匹配文件：检查 OLD 是否与文件名完全一致（区分大小写），或把 ROOT 改成有文件的目录。")
elif DO_RENAME:
    for p in matches:
        new_name = NEW + p.name[len(OLD):]
        dest = p.with_name(new_name)
        if dest.exists() and dest.resolve() != p.resolve():
            print("SKIP (已存在):", dest)
            continue
        p.rename(dest)
        print("RENAMED:", p.name, "->", new_name)
    print("完成。")
else:
    print("\nDO_RENAME=False，未修改。确认列表无误后设 DO_RENAME=True 再运行。")

/content/ImmuScope
CWD: /content/ImmuScope
ROOT: /content/ImmuScope
Files with 'A2-no-si' in name: 0

Files with name.startswith('ImmuScope-EL-no-si'): 45
  configs/ImmuScope-EL-no-si-A3-supcon.yaml
  configs/ImmuScope-EL-no-si-A4-no-metric.yaml
  configs/ImmuScope-EL-no-si-no-ftb-A3-supcon.yaml
  configs/ImmuScope-EL-no-si-no-ftb-A4-no-metric.yaml
  configs/ImmuScope-EL-no-si-no-ftb.yaml
  configs/ImmuScope-EL-no-si.yaml
  results/logs/ImmuScope-EL-no-si-A3-supcon_26-04-04_14-17-35-debug.log
  results/logs/ImmuScope-EL-no-si-A3-supcon_26-04-04_14-17-35-info.log
  results/logs/ImmuScope-EL-no-si-A3-supcon_26-04-05_02-54-40-debug.log
  results/logs/ImmuScope-EL-no-si-A3-supcon_26-04-05_02-54-40-info.log
  results/logs/ImmuScope-EL-no-si-A3-supcon_26-04-05_03-00-24-debug.log
  results/logs/ImmuScope-EL-no-si-A3-supcon_26-04-05_03-00-24-info.log
  results/logs/ImmuScope-EL-no-si-A3-supcon_26-04-05_06-16-04-debug.log
  results/logs/ImmuScope-EL-no-si-A3-supcon_26-04-05_06-16-04-info.log
  

In [ ]:
# w/o self-iterative（新版本ImmuScope-EL-no-si仅包含w/o SI）
!python -u main_antigen_presentation_train.py \
  --data-cnf configs/data.yaml \
  --model-cnf configs/ImmuScope-EL-no-si.yaml\
  --start-id 0 --num_models 1

In [ ]:
# w/o self-iterative & w/o fine-tune-b (旧版本ImmuScope-EL-no-si同时包含w/o SI和FTB，生成文件已修改文件名)
!python -u main_antigen_presentation_train_dongyizhe_20260405.py \
  --data-cnf configs/data.yaml \
  --model-cnf configs/ImmuScope-EL-no-si-no-ftb.yaml\
  --start-id 0 --num_models 1

[I 260405 10:09 utils:64] Model Name: ImmuScope-EL-no-si-no-ftb
[I 260405 10:09 main_antigen_presentation_train_dongyizhe_20260405:97] ------------- Start training model_id: 0 -  ------------
Training: 100% 12468/12468 [21:38<00:00,  9.60it/s]
[D 260405 10:31 trainer_el:190]  ============== Valid Bag: AUC0_1: 0.0827 AUPR: 0.8523 PPV: 0.7881
[D 260405 10:32 trainer_el:211]  ============== Test Bag: AUC0_1: 0.0796 AUPR: 0.8313 PPV: 0.7716
[I 260405 10:32 trainer_el:451] Epoch-pretrain: 0 - Loss[SA_I 0.1206 SA_B 0.1233 MA 0.2494 Tri 0.0685] -VAL 0.2169 - Valid:[AUC0_1:0.0826 AUPR:0.8529 PPV:0.7880] -- Test: AUPR: 0.8561 -Group [AUC0_1: 0.0794, AUPR: 0.8330, PPV: 0.7720]
Training: 100% 12468/12468 [21:24<00:00,  9.71it/s]
[D 260405 10:53 trainer_el:190]  ============== Valid Bag: AUC0_1: 0.0836 AUPR: 0.8575 PPV: 0.7976
[D 260405 10:54 trainer_el:211]  ============== Test Bag: AUC0_1: 0.0806 AUPR: 0.8378 PPV: 0.7794
[I 260405 10:54 trainer_el:451] Epoch-pretrain: 1 - Loss[SA_I 0.0803 SA_B 0

In [ ]:
# 保存到drive
%cd /content/ImmuScope

import shutil
import time
from pathlib import Path

PROJECT = Path("/content/ImmuScope")
BACKUP = Path("/content/drive/MyDrive/ImmuScope_ablation_backup")
BACKUP.mkdir(parents=True, exist_ok=True)

for name in ("weights", "results"):
    src = PROJECT / name
    if src.exists():
        dst = BACKUP / name
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print("Copied", src, "->", dst)
    else:
        print("Skip (missing):", src)
print(f"{time.strftime('%H:%M:%S')} 备份完成，Backup root:{BACKUP}")

/content/ImmuScope
Copied /content/ImmuScope/weights -> /content/drive/MyDrive/ImmuScope_ablation_backup/weights
Copied /content/ImmuScope/results -> /content/drive/MyDrive/ImmuScope_ablation_backup/results
17:30:18 备份完成，Backup root:/content/drive/MyDrive/ImmuScope_ablation_backup


In [ ]:
# Triplet → SupCon
!python -u main_antigen_presentation_train_dongyizhe_20260405.py \
  --data-cnf configs/data.yaml \
  --model-cnf configs/ImmuScope-EL-no-si-no-ftb-A3-supcon.yaml\
  --start-id 0 --num_models 1

[I 260405 17:30 utils:64] Model Name: ImmuScope-EL-no-si-no-ftb-A3-supcon
[I 260405 17:30 main_antigen_presentation_train_dongyizhe_20260405:97] ------------- Start training model_id: 0 -  ------------
Training: 100% 12468/12468 [21:35<00:00,  9.63it/s]
[D 260405 17:53 trainer_el:190]  ============== Valid Bag: AUC0_1: 0.0818 AUPR: 0.8433 PPV: 0.7796
[D 260405 17:53 trainer_el:211]  ============== Test Bag: AUC0_1: 0.0785 AUPR: 0.8233 PPV: 0.7645
[I 260405 17:53 trainer_el:451] Epoch-pretrain: 0 - Loss[SA_I 0.1276 SA_B 0.1301 MA 0.2559 Tri 0.0688] -VAL 0.2230 - Valid:[AUC0_1:0.0818 AUPR:0.8436 PPV:0.7794] -- Test: AUPR: 0.8481 -Group [AUC0_1: 0.0786, AUPR: 0.8245, PPV: 0.7646]
Training: 100% 12468/12468 [21:21<00:00,  9.73it/s]
[D 260405 18:15 trainer_el:190]  ============== Valid Bag: AUC0_1: 0.0835 AUPR: 0.8597 PPV: 0.7963
[D 260405 18:15 trainer_el:211]  ============== Test Bag: AUC0_1: 0.0802 AUPR: 0.8386 PPV: 0.7801
[I 260405 18:15 trainer_el:451] Epoch-pretrain: 1 - Loss[SA_I 0.0

In [ ]:
# 保存到drive
%cd /content/ImmuScope

import shutil
import time
from pathlib import Path

PROJECT = Path("/content/ImmuScope")
BACKUP = Path("/content/drive/MyDrive/ImmuScope_ablation_backup")
BACKUP.mkdir(parents=True, exist_ok=True)

for name in ("weights", "results"):
    src = PROJECT / name
    if src.exists():
        dst = BACKUP / name
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print("Copied", src, "->", dst)
    else:
        print("Skip (missing):", src)
print(f"{time.strftime('%H:%M:%S')} 备份完成，Backup root:{BACKUP}")

In [ ]:
# w/o triplet 度量项
!python -u main_antigen_presentation_train_dongyizhe_20260405.py \
  --data-cnf configs/data.yaml \
  --model-cnf configs/ImmuScope-EL-no-si-no-ftb-A4-no-metric.yaml\
  --start-id 0 --num_models 1

[I 260406 05:58 utils:64] Model Name: ImmuScope-EL-no-si-no-ftb-A4-no-metric
[I 260406 05:58 main_antigen_presentation_train_dongyizhe_20260405:106] ------------- Start training model_id: 0 -  ------------
Training: 100% 12468/12468 [20:44<00:00, 10.02it/s]
[D 260406 06:20 trainer_el:190]  ============== Valid Bag: AUC0_1: 0.0824 AUPR: 0.8496 PPV: 0.7847
[D 260406 06:21 trainer_el:211]  ============== Test Bag: AUC0_1: 0.0791 AUPR: 0.8257 PPV: 0.7687
[I 260406 06:21 trainer_el:451] Epoch-pretrain: 0 - Loss[SA_I 0.1232 SA_B 0.1254 MA 0.2489 Tri 0.0687] -VAL 0.2169 - Valid:[AUC0_1:0.0827 AUPR:0.8521 PPV:0.7842] -- Test: AUPR: 0.8545 -Group [AUC0_1: 0.0792, AUPR: 0.8294, PPV: 0.7686]
Training: 100% 12468/12468 [20:41<00:00, 10.04it/s]
[D 260406 06:42 trainer_el:190]  ============== Valid Bag: AUC0_1: 0.0838 AUPR: 0.8593 PPV: 0.7986
[D 260406 06:42 trainer_el:211]  ============== Test Bag: AUC0_1: 0.0806 AUPR: 0.8401 PPV: 0.7845
[I 260406 06:42 trainer_el:451] Epoch-pretrain: 1 - Loss[SA_I

In [ ]:
# 保存到drive
%cd /content/ImmuScope

import shutil
import time
from pathlib import Path

PROJECT = Path("/content/ImmuScope")
BACKUP = Path("/content/drive/MyDrive/ImmuScope_ablation_backup/A4")
BACKUP.mkdir(parents=True, exist_ok=True)

INTERVAL_SECONDS = 600 # 按需修改
print(f"开始每 {INTERVAL_SECONDS} 秒备份一次，中断请点单元格的停止按钮。")

while True:
    for name in ("weights", "results"):
        src = PROJECT / name
        if src.exists():
            dst = BACKUP / name
            if dst.exists():
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
            print("Copied", src, "->", dst)
        else:
            print("Skip (missing):", src)
    print(f"{time.strftime('%H:%M:%S')} 备份完成，Backup root:{BACKUP}，{INTERVAL_SECONDS}s 后下次备份...")
    time.sleep(INTERVAL_SECONDS)

/content/ImmuScope
开始每 600 秒备份一次，中断请点单元格的停止按钮。
Copied /content/ImmuScope/weights -> /content/drive/MyDrive/ImmuScope_ablation_backup/A4/weights
Copied /content/ImmuScope/results -> /content/drive/MyDrive/ImmuScope_ablation_backup/A4/results
13:08:36 备份完成，Backup root:/content/drive/MyDrive/ImmuScope_ablation_backup/A4，600s 后下次备份...
Copied /content/ImmuScope/weights -> /content/drive/MyDrive/ImmuScope_ablation_backup/A4/weights
Copied /content/ImmuScope/results -> /content/drive/MyDrive/ImmuScope_ablation_backup/A4/results
13:18:38 备份完成，Backup root:/content/drive/MyDrive/ImmuScope_ablation_backup/A4，600s 后下次备份...
Copied /content/ImmuScope/weights -> /content/drive/MyDrive/ImmuScope_ablation_backup/A4/weights
Copied /content/ImmuScope/results -> /content/drive/MyDrive/ImmuScope_ablation_backup/A4/results
13:28:39 备份完成，Backup root:/content/drive/MyDrive/ImmuScope_ablation_backup/A4，600s 后下次备份...
Copied /content/ImmuScope/weights -> /content/drive/MyDrive/ImmuScope_ablation_backup/A4/weight

KeyboardInterrupt: 

In [ ]:
# 从drive中引入消融权重
import os
import shutil
from pathlib import Path

def copy_to_local_EL(DRIVE_EL_DIR, LOCAL_EL):
  for p in DRIVE_EL_DIR.glob("*.pt"):
    dest = LOCAL_EL / p.name
    shutil.copy2(p, dest)
    print("Copied", p.name, "->", dest)

#A2
DRIVE_EL_DIR_A2 = Path("/content/drive/MyDrive/ImmuScope_ablation_backup/A2/weights/EL")

LOCAL_EL_A2 = Path("/content/ImmuScope/weights/EL")
LOCAL_EL_A2.mkdir(parents=True, exist_ok=True)
copy_to_local_EL(DRIVE_EL_DIR_A2, LOCAL_EL_A2)
print("imported A2 weights")

#A3
DRIVE_EL_DIR_A3 = Path("/content/drive/MyDrive/ImmuScope_ablation_backup/A3/weights/EL")

LOCAL_EL_A3 = Path("/content/ImmuScope/weights/EL")
LOCAL_EL_A3.mkdir(parents=True, exist_ok=True)
copy_to_local_EL(DRIVE_EL_DIR_A3, LOCAL_EL_A3)
print("imported A3 weights")

#A4
DRIVE_EL_DIR_A4 = Path("/content/drive/MyDrive/ImmuScope_ablation_backup/A4/weights/EL")

LOCAL_EL_A4 = Path("/content/ImmuScope/weights/EL")
LOCAL_EL_A4.mkdir(parents=True, exist_ok=True)
copy_to_local_EL(DRIVE_EL_DIR_A4, LOCAL_EL_A4)
print("imported A4 weights")

Copied ImmuScope-EL-9-fine-tune-b.pt -> /content/ImmuScope/weights/EL/ImmuScope-EL-9-fine-tune-b.pt
Copied ImmuScope-EL-1-fine-tune-b.pt -> /content/ImmuScope/weights/EL/ImmuScope-EL-1-fine-tune-b.pt
Copied ImmuScope-EL-2-pretrain.pt -> /content/ImmuScope/weights/EL/ImmuScope-EL-2-pretrain.pt
Copied ImmuScope-EL-8-fine-tune-b.pt -> /content/ImmuScope/weights/EL/ImmuScope-EL-8-fine-tune-b.pt
Copied ImmuScope-EL-0-pretrain.pt -> /content/ImmuScope/weights/EL/ImmuScope-EL-0-pretrain.pt
Copied ImmuScope-EL-2-fine-tune-b.pt -> /content/ImmuScope/weights/EL/ImmuScope-EL-2-fine-tune-b.pt
Copied ImmuScope-EL-6-pretrain.pt -> /content/ImmuScope/weights/EL/ImmuScope-EL-6-pretrain.pt
Copied ImmuScope-EL-1-pretrain.pt -> /content/ImmuScope/weights/EL/ImmuScope-EL-1-pretrain.pt
Copied ImmuScope-EL-9-pretrain.pt -> /content/ImmuScope/weights/EL/ImmuScope-EL-9-pretrain.pt
Copied ImmuScope-EL-0-fine-tune-b.pt -> /content/ImmuScope/weights/EL/ImmuScope-EL-0-fine-tune-b.pt
Copied ImmuScope-EL-4-fine-tun

In [ ]:
# 在A2消融下游 CD4 training
%cd /content/ImmuScope
!python main_cd4_epitope_train_dongyizhe_20260405.py \
  --data-cnf configs/data.yaml \
  --model-cnf configs/ImmuScope.yaml \
  --el-stem ImmuScope-EL-no-si-no-ftb \
  --el-suffix pretrain \
  --weights-tag from-EL-A2 \
  --start-id 0 --num_models 1

/content/ImmuScope
[I 260406 17:13 utils:64] Model Name: ImmuScope
[I 260406 17:13 main_cd4_epitope_train_dongyizhe_20260405:114] Loading EL init from weights/EL/ImmuScope-EL-no-si-no-ftb-0-pretrain.pt
[I 260406 17:13 main_cd4_epitope_train_dongyizhe_20260405:43] Start training model weights/CD4/ImmuScope-from-EL-A2-0.pt
[I 260406 17:13 trainer_cd4_epitope:88] ==== Model loaded from weights/EL/ImmuScope-EL-no-si-no-ftb-0-pretrain.pt ====
[I 260406 17:14 trainer_cd4_epitope:164] Epoch: 0 - Loss: BA 0.05995, SA 0.00000 - Valid :[AUC0_1:0.0435 AUPR:0.7345 PPV:0.6620]
[I 260406 17:14 trainer_cd4_epitope:164] Epoch: 1 - Loss: BA 0.04244, SA 0.00000 - Valid :[AUC0_1:0.0453 AUPR:0.7510 PPV:0.6767]
[I 260406 17:14 trainer_cd4_epitope:164] Epoch: 2 - Loss: BA 0.03913, SA 0.00000 - Valid :[AUC0_1:0.0465 AUPR:0.7580 PPV:0.6835]
[I 260406 17:15 trainer_cd4_epitope:164] Epoch: 3 - Loss: BA 0.03736, SA 0.00000 - Valid :[AUC0_1:0.0468 AUPR:0.7584 PPV:0.6867]
[I 260406 17:15 trainer_cd4_epitope:164] E

In [ ]:
# A2-CD4 test
%cd /content/ImmuScope
!python main_cd4_epitope_test_dongyizhe_20260405.py \
  --data-cnf configs/data.yaml \
  --model-cnf configs/ImmuScope.yaml \
  --weights-tag from-EL-A2 \
  --start-id 0 --num_models 1

/content/ImmuScope
[I 260406 17:23 utils:64] Model Name: ImmuScope
[I 260406 17:23 trainer_cd4_epitope:88] ==== Model loaded from weights/CD4/ImmuScope-from-EL-A2-0.pt ====
[I 260406 17:24 main_cd4_epitope_test_dongyizhe_20260405:56] |**---------Model 0--- TEST: Median AUC: 0.8988; Mean AUC: 0.8038; AVG AUC: 0.7481---------**|
[I 260406 17:24 main_cd4_epitope_test_dongyizhe_20260405:62] -----------------Average-----------------
[I 260406 17:24 main_cd4_epitope_test_dongyizhe_20260405:74] |**========== TEST: Median AUC: 0.8988; Mean AUC: 0.8038; AVG AUC: 0.7481==========**|


In [ ]:
# 在A2消融下游IM training
%cd /content/ImmuScope
!python main_immunogenicity_train_dongyizhe_20260405.py \
  --data-cnf configs/data.yaml \
  --model-cnf configs/ImmuScope-IM.yaml \
  --el-stem ImmuScope-EL-no-si-no-ftb \
  --el-suffix pretrain \
  --weights-tag from-EL-A2 \
  --start-id 0 --num_models 1

/content/ImmuScope
[I 260406 17:24 utils:64] Model Name: ImmuScope-IM
[I 260406 17:24 main_immunogenicity_train_dongyizhe_20260405:100] Loading EL init from weights/EL/ImmuScope-EL-no-si-no-ftb-0-pretrain.pt
[I 260406 17:24 main_immunogenicity_train_dongyizhe_20260405:43] Start training model weights/IM/ImmuScope-IM-from-EL-A2-0.pt
[I 260406 17:24 trainer_immunogenicity:84] ==== Model loaded from weights/EL/ImmuScope-EL-no-si-no-ftb-0-pretrain.pt ====
Training: 100% 504/504 [00:10<00:00, 46.78it/s]
[I 260406 17:24 trainer_immunogenicity:113] Epoch: 0 - Loss: IMM 0.23817  - Valid-AUC:0.8363  -- Test: [AUC-Group: 0.8278 - AUC-All: 0.8423]
Training: 100% 504/504 [00:10<00:00, 49.00it/s]
[I 260406 17:24 trainer_immunogenicity:113] Epoch: 1 - Loss: IMM 0.21386  - Valid-AUC:0.8439  -- Test: [AUC-Group: 0.8380 - AUC-All: 0.8512]
Training: 100% 504/504 [00:10<00:00, 49.23it/s]
[I 260406 17:24 trainer_immunogenicity:113] Epoch: 2 - Loss: IMM 0.20153  - Valid-AUC:0.8487  -- Test: [AUC-Group: 0.8

In [ ]:
# A2-IM test
%cd /content/ImmuScope
!python main_immunogenicity_test_dongyizhe_20260405.py \
  --data-cnf configs/data.yaml \
  --model-cnf configs/ImmuScope.yaml \
  --weights-tag IM-from-EL-A2 \
  --start-id 0 --num_models 1

/content/ImmuScope
[I 260406 17:38 utils:64] Model Name: ImmuScope
[I 260406 17:38 trainer_immunogenicity:84] ==== Model loaded from weights/IM/ImmuScope-IM-from-EL-A2-0.pt ====
[I 260406 17:38 main_immunogenicity_test_dongyizhe_20260405:77] |**TEST: AUC_GROUP: 0.8685**|
[I 260406 17:38 main_immunogenicity_test_dongyizhe_20260405:78] |**TEST: AUC_ALL: 0.8667**|
[I 260406 17:38 main_immunogenicity_test_dongyizhe_20260405:82] -----------------Average-----------------
[I 260406 17:38 main_immunogenicity_test_dongyizhe_20260405:92] |**========== TEST: AUC_GROUP: 0.8685 =========**|
[I 260406 17:38 main_immunogenicity_test_dongyizhe_20260405:93] |**========== TEST: AUC_ALL: 0.8667 =========**|


In [ ]:
# A2下游保存到drive
%cd /content/ImmuScope

import shutil
import time
from pathlib import Path

BACKUP = Path("/content/drive/MyDrive/ImmuScope_ablation_backup/A2")
BACKUP.mkdir(parents=True, exist_ok=True)

!cp -rn /content/ImmuScope/weights/. "/content/drive/MyDrive/ImmuScope_ablation_backup/A2/weights/"
!cp -rn /content/ImmuScope/results/. "/content/drive/MyDrive/ImmuScope_ablation_backup/A2/results/"

print(f"备份完成，Backup root:{BACKUP}")

/content/ImmuScope
备份完成，Backup root:/content/drive/MyDrive/ImmuScope_ablation_backup/A2


In [ ]:
# 在A3消融下游 CD4 training
%cd /content/ImmuScope
!python main_cd4_epitope_train_dongyizhe_20260405.py \
  --data-cnf configs/data.yaml \
  --model-cnf configs/ImmuScope.yaml \
  --el-stem ImmuScope-EL-no-si-no-ftb-A3-supcon \
  --el-suffix pretrain \
  --weights-tag from-EL-A3 \
  --start-id 0 --num_models 1

/content/ImmuScope
[I 260406 17:38 utils:64] Model Name: ImmuScope
[I 260406 17:38 main_cd4_epitope_train_dongyizhe_20260405:114] Loading EL init from weights/EL/ImmuScope-EL-no-si-no-ftb-A3-supcon-0-pretrain.pt
[I 260406 17:38 main_cd4_epitope_train_dongyizhe_20260405:43] Start training model weights/CD4/ImmuScope-from-EL-A3-0.pt
[I 260406 17:38 trainer_cd4_epitope:88] ==== Model loaded from weights/EL/ImmuScope-EL-no-si-no-ftb-A3-supcon-0-pretrain.pt ====
[I 260406 17:38 trainer_cd4_epitope:164] Epoch: 0 - Loss: BA 0.06745, SA 0.00000 - Valid :[AUC0_1:0.0417 AUPR:0.7231 PPV:0.6585]
[I 260406 17:39 trainer_cd4_epitope:164] Epoch: 1 - Loss: BA 0.04525, SA 0.00000 - Valid :[AUC0_1:0.0445 AUPR:0.7466 PPV:0.6753]
[I 260406 17:39 trainer_cd4_epitope:164] Epoch: 2 - Loss: BA 0.04158, SA 0.00000 - Valid :[AUC0_1:0.0458 AUPR:0.7560 PPV:0.6809]
[I 260406 17:40 trainer_cd4_epitope:164] Epoch: 3 - Loss: BA 0.03943, SA 0.00000 - Valid :[AUC0_1:0.0467 AUPR:0.7600 PPV:0.6814]
[I 260406 17:40 traine

In [ ]:
# A3-CD4 test
%cd /content/ImmuScope
!python main_cd4_epitope_test_dongyizhe_20260405.py \
  --data-cnf configs/data.yaml \
  --model-cnf configs/ImmuScope.yaml \
  --weights-tag from-EL-A3 \
  --start-id 0 --num_models 1

/content/ImmuScope
[I 260406 17:48 utils:64] Model Name: ImmuScope
[I 260406 17:48 trainer_cd4_epitope:88] ==== Model loaded from weights/CD4/ImmuScope-from-EL-A3-0.pt ====
[I 260406 17:48 main_cd4_epitope_test_dongyizhe_20260405:56] |**---------Model 0--- TEST: Median AUC: 0.9078; Mean AUC: 0.8056; AVG AUC: 0.7517---------**|
[I 260406 17:48 main_cd4_epitope_test_dongyizhe_20260405:62] -----------------Average-----------------
[I 260406 17:48 main_cd4_epitope_test_dongyizhe_20260405:74] |**========== TEST: Median AUC: 0.9078; Mean AUC: 0.8056; AVG AUC: 0.7517==========**|


In [ ]:
# 在A3消融下游IM training
%cd /content/ImmuScope
!python main_immunogenicity_train_dongyizhe_20260405.py \
  --data-cnf configs/data.yaml \
  --model-cnf configs/ImmuScope-IM.yaml \
  --el-stem ImmuScope-EL-no-si-no-ftb-A3-supcon \
  --el-suffix pretrain \
  --weights-tag from-EL-A3 \
  --start-id 0 --num_models 1

/content/ImmuScope
[I 260406 17:49 utils:64] Model Name: ImmuScope-IM
[I 260406 17:49 main_immunogenicity_train_dongyizhe_20260405:100] Loading EL init from weights/EL/ImmuScope-EL-no-si-no-ftb-A3-supcon-0-pretrain.pt
[I 260406 17:49 main_immunogenicity_train_dongyizhe_20260405:43] Start training model weights/IM/ImmuScope-IM-from-EL-A3-0.pt
[I 260406 17:49 trainer_immunogenicity:84] ==== Model loaded from weights/EL/ImmuScope-EL-no-si-no-ftb-A3-supcon-0-pretrain.pt ====
Training: 100% 504/504 [00:10<00:00, 49.67it/s]
[I 260406 17:49 trainer_immunogenicity:113] Epoch: 0 - Loss: IMM 0.23947  - Valid-AUC:0.8204  -- Test: [AUC-Group: 0.8141 - AUC-All: 0.8295]
Training: 100% 504/504 [00:09<00:00, 52.92it/s]
[I 260406 17:49 trainer_immunogenicity:113] Epoch: 1 - Loss: IMM 0.21621  - Valid-AUC:0.8323  -- Test: [AUC-Group: 0.8192 - AUC-All: 0.8417]
Training: 100% 504/504 [00:09<00:00, 51.89it/s]
[I 260406 17:49 trainer_immunogenicity:113] Epoch: 2 - Loss: IMM 0.20261  - Valid-AUC:0.8465  -- T

In [ ]:
# A3-IM test
%cd /content/ImmuScope
!python main_immunogenicity_test_dongyizhe_20260405.py \
  --data-cnf configs/data.yaml \
  --model-cnf configs/ImmuScope.yaml \
  --weights-tag IM-from-EL-A3 \
  --start-id 0 --num_models 1

/content/ImmuScope
[I 260406 17:52 utils:64] Model Name: ImmuScope
[I 260406 17:52 trainer_immunogenicity:84] ==== Model loaded from weights/IM/ImmuScope-IM-from-EL-A3-0.pt ====
[I 260406 17:52 main_immunogenicity_test_dongyizhe_20260405:77] |**TEST: AUC_GROUP: 0.8722**|
[I 260406 17:52 main_immunogenicity_test_dongyizhe_20260405:78] |**TEST: AUC_ALL: 0.8649**|
[I 260406 17:52 main_immunogenicity_test_dongyizhe_20260405:82] -----------------Average-----------------
[I 260406 17:52 main_immunogenicity_test_dongyizhe_20260405:92] |**========== TEST: AUC_GROUP: 0.8722 =========**|
[I 260406 17:52 main_immunogenicity_test_dongyizhe_20260405:93] |**========== TEST: AUC_ALL: 0.8649 =========**|


In [ ]:
# A3下游保存到drive
%cd /content/ImmuScope

import shutil
import time
from pathlib import Path

BACKUP = Path("/content/drive/MyDrive/ImmuScope_ablation_backup/A3")
BACKUP.mkdir(parents=True, exist_ok=True)

!cp -rn /content/ImmuScope/weights/. "/content/drive/MyDrive/ImmuScope_ablation_backup/A3/weights/"
!cp -rn /content/ImmuScope/results/. "/content/drive/MyDrive/ImmuScope_ablation_backup/A3/results/"

print(f"备份完成，Backup root:{BACKUP}")

/content/ImmuScope
备份完成，Backup root:/content/drive/MyDrive/ImmuScope_ablation_backup/A3


In [ ]:
# 在A4消融下游 CD4 training
%cd /content/ImmuScope
!python main_cd4_epitope_train_dongyizhe_20260405.py \
  --data-cnf configs/data.yaml \
  --model-cnf configs/ImmuScope.yaml \
  --el-stem ImmuScope-EL-no-si-no-ftb-A4-no-metric \
  --el-suffix pretrain \
  --weights-tag from-EL-A4 \
  --start-id 0 --num_models 1

/content/ImmuScope
[I 260406 17:52 utils:64] Model Name: ImmuScope
[I 260406 17:52 main_cd4_epitope_train_dongyizhe_20260405:114] Loading EL init from weights/EL/ImmuScope-EL-no-si-no-ftb-A4-no-metric-0-pretrain.pt
[I 260406 17:52 main_cd4_epitope_train_dongyizhe_20260405:43] Start training model weights/CD4/ImmuScope-from-EL-A4-0.pt
[I 260406 17:52 trainer_cd4_epitope:88] ==== Model loaded from weights/EL/ImmuScope-EL-no-si-no-ftb-A4-no-metric-0-pretrain.pt ====
[I 260406 17:53 trainer_cd4_epitope:164] Epoch: 0 - Loss: BA 0.06254, SA 0.00000 - Valid :[AUC0_1:0.0442 AUPR:0.7417 PPV:0.6684]
[I 260406 17:53 trainer_cd4_epitope:164] Epoch: 1 - Loss: BA 0.04291, SA 0.00000 - Valid :[AUC0_1:0.0463 AUPR:0.7598 PPV:0.6822]
[I 260406 17:54 trainer_cd4_epitope:164] Epoch: 2 - Loss: BA 0.04000, SA 0.00000 - Valid :[AUC0_1:0.0472 AUPR:0.7640 PPV:0.6863]
[I 260406 17:54 trainer_cd4_epitope:164] Epoch: 3 - Loss: BA 0.03812, SA 0.00000 - Valid :[AUC0_1:0.0474 AUPR:0.7640 PPV:0.6860]
[I 260406 17:55 

In [ ]:
# A4-CD4 test
%cd /content/ImmuScope
!python main_cd4_epitope_test_dongyizhe_20260405.py \
  --data-cnf configs/data.yaml \
  --model-cnf configs/ImmuScope.yaml \
  --weights-tag from-EL-A4 \
  --start-id 0 --num_models 1

/content/ImmuScope
[I 260406 18:02 utils:64] Model Name: ImmuScope
[I 260406 18:02 trainer_cd4_epitope:88] ==== Model loaded from weights/CD4/ImmuScope-from-EL-A4-0.pt ====
[I 260406 18:03 main_cd4_epitope_test_dongyizhe_20260405:56] |**---------Model 0--- TEST: Median AUC: 0.9018; Mean AUC: 0.8073; AVG AUC: 0.7521---------**|
[I 260406 18:03 main_cd4_epitope_test_dongyizhe_20260405:62] -----------------Average-----------------
[I 260406 18:03 main_cd4_epitope_test_dongyizhe_20260405:74] |**========== TEST: Median AUC: 0.9018; Mean AUC: 0.8073; AVG AUC: 0.7521==========**|


In [ ]:
# 在A4消融下游IM training
%cd /content/ImmuScope
!python main_immunogenicity_train_dongyizhe_20260405.py \
  --data-cnf configs/data.yaml \
  --model-cnf configs/ImmuScope-IM.yaml \
  --el-stem ImmuScope-EL-no-si-no-ftb-A4-no-metric \
  --el-suffix pretrain \
  --weights-tag from-EL-A4 \
  --start-id 0 --num_models 1

/content/ImmuScope
[I 260407 01:55 utils:64] Model Name: ImmuScope-IM
[I 260407 01:55 main_immunogenicity_train_dongyizhe_20260405:100] Loading EL init from weights/EL/ImmuScope-EL-no-si-no-ftb-A4-no-metric-0-pretrain.pt
[I 260407 01:55 main_immunogenicity_train_dongyizhe_20260405:43] Start training model weights/IM/ImmuScope-IM-from-EL-A4-0.pt
[I 260407 01:55 trainer_immunogenicity:84] ==== Model loaded from weights/EL/ImmuScope-EL-no-si-no-ftb-A4-no-metric-0-pretrain.pt ====
Training: 100% 504/504 [00:10<00:00, 49.32it/s]
[I 260407 01:55 trainer_immunogenicity:113] Epoch: 0 - Loss: IMM 0.23662  - Valid-AUC:0.8339  -- Test: [AUC-Group: 0.8103 - AUC-All: 0.8348]
Training: 100% 504/504 [00:09<00:00, 52.46it/s]
[I 260407 01:55 trainer_immunogenicity:113] Epoch: 1 - Loss: IMM 0.21102  - Valid-AUC:0.8491  -- Test: [AUC-Group: 0.8322 - AUC-All: 0.8476]
Training: 100% 504/504 [00:09<00:00, 51.81it/s]
[I 260407 01:56 trainer_immunogenicity:113] Epoch: 2 - Loss: IMM 0.19731  - Valid-AUC:0.8476

In [ ]:
# A4-IM test
%cd /content/ImmuScope
!python main_immunogenicity_test_dongyizhe_20260405.py \
  --data-cnf configs/data.yaml \
  --model-cnf configs/ImmuScope.yaml \
  --weights-tag IM-from-EL-A4 \
  --start-id 0 --num_models 1

/content/ImmuScope
[I 260407 01:59 utils:64] Model Name: ImmuScope
[I 260407 01:59 trainer_immunogenicity:84] ==== Model loaded from weights/IM/ImmuScope-IM-from-EL-A4-0.pt ====
[I 260407 01:59 main_immunogenicity_test_dongyizhe_20260405:77] |**TEST: AUC_GROUP: 0.8603**|
[I 260407 01:59 main_immunogenicity_test_dongyizhe_20260405:78] |**TEST: AUC_ALL: 0.8641**|
[I 260407 01:59 main_immunogenicity_test_dongyizhe_20260405:82] -----------------Average-----------------
[I 260407 01:59 main_immunogenicity_test_dongyizhe_20260405:92] |**========== TEST: AUC_GROUP: 0.8603 =========**|
[I 260407 01:59 main_immunogenicity_test_dongyizhe_20260405:93] |**========== TEST: AUC_ALL: 0.8641 =========**|


In [ ]:
# A4下游保存到drive
%cd /content/ImmuScope

import shutil
import time
from pathlib import Path

BACKUP = Path("/content/drive/MyDrive/ImmuScope_ablation_backup/A4")
BACKUP.mkdir(parents=True, exist_ok=True)

!cp -rn /content/ImmuScope/weights/. "/content/drive/MyDrive/ImmuScope_ablation_backup/A4/weights/"
!cp -rn /content/ImmuScope/results/. "/content/drive/MyDrive/ImmuScope_ablation_backup/A4/results/"

print(f"备份完成，Backup root:{BACKUP}")

/content/ImmuScope
备份完成，Backup root:/content/drive/MyDrive/ImmuScope_ablation_backup/A4


In [ ]:
# 防runtime exceeded

import time
interval = 600
while True:
  time.sleep(interval)

KeyboardInterrupt: 